In [ ]:
import pandas as pd
from pathlib import Path

FILE_PATH = Path("Copy of Master Data_290102026 2 - Copy (wecompress.com).xlsx")

# Load category mapping
cat_df = pd.read_excel(FILE_PATH, sheet_name="Sheet1")
cat_df = cat_df[["PARTNO", "Category"]].dropna(subset=["PARTNO"])
cat_map = dict(zip(cat_df["PARTNO"], cat_df["Category"]))

# Load master data
master = pd.read_excel(FILE_PATH, sheet_name="Master Data ")

for col in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity", "Cycle Time", "Tonnage"]:
    if col in master.columns:
        master[col] = pd.to_numeric(master[col], errors="coerce").fillna(0)

# Aggregate only 120T machines
records = []
for child, g in master.groupby("Child Part"):
    # Only if has 120T machine
    if not (g["Tonnage"] == 120).any():
        continue

    daily_demand = (g["Daily Plan"] * g["Sub Count"]).sum()
    if daily_demand <= 0:
        continue

    net_req = daily_demand + g["Minimum Quantity"].iloc[0] - g["Inventory_25"].iloc[0]
    if net_req <= 0:
        continue

    cycle_valid = g["Cycle Time"][g["Cycle Time"] > 0]
    if cycle_valid.empty:
        continue
    cycle_sec = cycle_valid.iloc[0]

    machines = list(set(normalize_machine(x) for x in g["Vertical Machines"].dropna().unique() if normalize_machine(x)))
    if not machines:
        continue

    records.append({
        "Child_Part": child,
        "Daily_Demand": daily_demand,
        "Net_Required": net_req,
        "Cycle_Time_sec": cycle_sec,
        "Eligible_Machines": machines,
        "Category": cat_map.get(child, "Unknown"),
        "Inventory": g["Inventory_25"].iloc[0]
    })

df_ml = pd.DataFrame(records)
print(f"ML-ready dataset created: {len(df_ml):,} rows (120T machines only)")
print(df_ml["Category"].value_counts())

# Save for next steps
df_ml.to_csv("120t_ml_ready.csv", index=False)
print("Saved → 120t_ml_ready.csv")

In [ ]:
import numpy as np
import random

# Number of simulated days
SIM_DAYS = 30

# Create simulation log
sim_data = []

# Current state (from real data)
current_inventory = dict(zip(df_ml["Child_Part"], df_ml["Inventory"]))
last_prod_day = {part: -1 for part in df_ml["Child_Part"]}  # -1 = never produced

for day in range(SIM_DAYS):
    print(f"Simulating day {day+1}/{SIM_DAYS}...", end="\r")
    
    # Slight demand noise
    day_demand_factor = np.random.uniform(0.8, 1.2, len(df_ml))
    df_day = df_ml.copy()
    df_day["Day_Demand"] = df_day["Daily_Demand"] * day_demand_factor
    
    # Calculate current coverage & urgency
    df_day["Coverage_Days"] = df_day["Child_Part"].map(current_inventory) / df_day["Daily_Demand"].clip(1)
    df_day["Days_Since_Last"] = day - df_day["Child_Part"].map(last_prod_day)
    df_day["Days_Since_Last"] = df_day["Days_Since_Last"].clip(lower=0)
    
    # Simple greedy-like selection (mimic your scheduler)
    selected = []
    remaining_capacity = 22.0 * 4  # rough total for 4 machines
    
    # Sort by urgency
    df_day = df_day.sort_values("Net_Required", ascending=False)
    
    for _, row in df_day.iterrows():
        if remaining_capacity <= 0:
            break
            
        prod_time = row["Cycle_Time_sec"] / 3600.0
        qty_possible = min(row["Net_Required"], remaining_capacity / prod_time)
        if qty_possible < 10:
            continue
            
        # Decide to produce (greedy logic)
        if qty_possible > 50 or row["Days_Since_Last"] > 15 or row["Coverage_Days"] < 5:
            selected.append(row["Child_Part"])
            produced_qty = min(qty_possible, row["Net_Required"])
            remaining_capacity -= produced_qty * prod_time + CHANGEOVER_HOURS
            
            # Update state
            current_inventory[row["Child_Part"]] -= produced_qty
            last_prod_day[row["Child_Part"]] = day
            
            # Label priority
            priority = 0.9 if row["Category"] == "Repeater" else 0.8
            if row["Days_Since_Last"] > 20:
                priority = 0.95  # force boost for neglected strangers
        else:
            priority = 0.3  # low urgency
            
        sim_data.append({
            "Day": day,
            "Child_Part": row["Child_Part"],
            "Category": row["Category"],
            "Daily_Demand": row["Daily_Demand"],
            "Net_Required": row["Net_Required"],
            "Inventory_Start_Day": current_inventory.get(row["Child_Part"], 0),
            "Days_Since_Last": row["Days_Since_Last"],
            "Produced_Today": 1 if row["Child_Part"] in selected else 0,
            "Priority_Label": priority
        })

# Create training DataFrame
df_train = pd.DataFrame(sim_data)
print(f"\nSimulation complete. Training data rows: {len(df_train):,}")

# Features for ML
features = [
    "Daily_Demand", "Net_Required", "Inventory_Start_Day",
    "Days_Since_Last", "Produced_Today"
]

X = df_train[features]
y = df_train["Priority_Label"]

print("\nSample of training data:")
print(df_train.head(10))

# Save for model training
df_train.to_csv("120t_ml_training_data.csv", index=False)
print("Training data saved → 120t_ml_training_data.csv")